# Data Exploration

This notebook documents the initial exploration of the Simpsons episodes dataset. The goal is to understand the data structure, identify quality issues and prepare the data cleaning.
Dataset: https://www.kaggle.com/datasets/prashant111/the-simpsons-dataset/data


## 1. Setup & Data Loading


In [ ]:
import pandas as pd
import altair as alt

In [ ]:
# Load and sort chronologically
data = pd.read_csv("../data/kaggle_data/simpsons_episodes.csv")
data = data.sort_values("number_in_series").reset_index(drop=True)
data.head(10)

## 2. Dataset Overview

The dataset contains one row per episode with the following columns:

| Column                   | Description                                                  |
| ------------------------ | ------------------------------------------------------------ |
| `id`                     | Episode identifier (same as `number_in_series`)              |
| `image_url`              | Link to episode thumbnail image (does not work)              |
| `imdb_rating`            | IMDb user rating (1-10 scale)                                |
| `imdb_votes`             | Number of IMDb user votes                                    |
| `number_in_season`       | Episode number within its season                             |
| `number_in_series`       | Overall episode number across all seasons                    |
| `original_air_date`      | Date of first broadcast (YYYY-MM-DD)                         |
| `original_air_year`      | Year of first broadcast                                      |
| `production_code`        | Internal production code                                     |
| `season`                 | Season number                                                |
| `title`                  | Episode title                                                |
| `us_viewers_in_millions` | U.S. television audience in millions                         |
| `video_url`              | Link to episode video                                        |
| `views`                  | View but not TV viewership (likely website views or similar) |


In [ ]:
data.info()

In [ ]:
data.describe()

In [ ]:
print(f"Shape: {data.shape[0]} episodes, {data.shape[1]} columns")
print(
    f"Seasons: {data['season'].min()} - {data['season'].max()} ({data['season'].nunique()} seasons)"
)
print(
    f"Air date range: {data['original_air_date'].min()} to {data['original_air_date'].max()}"
)

**Takeaway:** 600 episodes across 28 seasons, aired from 1989 to 2016.
Key columns for our analysis are `imdb_rating`, `us_viewers_in_millions`, `number_in_season`, `season`, and `original_air_date`.
Potentially of interest are: `imdb_votes`, `title`, `number_in_series`.

Important to distinguish: the `views` column does not match `us_viewers_in_millions` and likely represents website page views.

We can disregard and drop: `views`, `image_url`, `video_url`, `production_code`, `id` (identical to `number_in_series`), and `original_air_year` (derivable from `original_air_date`).


## 3. Missing Data


In [ ]:
# Count missing values per column
missing = data.isnull().sum()
missing[missing > 0]

In [ ]:
# Show all rows with any missing value
rows_with_nulls = data[data.isnull().any(axis=1)]
print(f"{len(rows_with_nulls)} rows have at least one missing value:\n")
rows_with_nulls[
    [
        "number_in_series",
        "season",
        "number_in_season",
        "title",
        "imdb_rating",
        "us_viewers_in_millions",
        "image_url",
        "views",
    ]
]

**Findings:**

- `us_viewers_in_millions`: 6 missing, with 3 from season 8 (episodes 160, 161, 173) and 3 from season 28 (episodes 598-600)
  - Missing values for season 8 can be looked up and filled in manually
- `imdb_rating` / `imdb_votes`: 3 missing, all from season 28 (episodes 598-600)
  - Data for season 28 is severely incomplete as it only includes 4 out of 22 episodes with missing data for ratings and views
  - Our best option is to drop all season 28 data and focus our analysis on seasons 1-27. Keeping it would skew per-season aggregates, since the 4 episodes are not representative of the full season, and 3 of them lack ratings and viewership entirely.
- `image_url` / `video_url` / `views`: 4 missing (episode 447 in season 21 and episodes 598-600 in season 28)
  - Can be disregarded, as these columns are not needed for our analysis and will be dropped during cleaning.


## 4. Duplicates


In [ ]:
print(f"Duplicate rows: {data.duplicated().sum()}")
print(f"Duplicate titles: {data['title'].duplicated().sum()}")
print(f"Duplicate number_in_series: {data['number_in_series'].duplicated().sum()}")

**Takeaway:** No duplicate rows, titles, or episode numbers. Each row represents a unique episode.


## 5. Data Types & Parsing


In [ ]:
# original_air_date is stored as string. check it parses correctly
print(f"original_air_date dtype: {data['original_air_date'].dtype}")

parsed_dates = pd.to_datetime(data["original_air_date"], format="%Y-%m-%d")
print(f"All dates parse correctly: range {parsed_dates.min()} to {parsed_dates.max()}")

In [ ]:
# Verify original_air_year matches the year in original_air_date
year_mismatch = data[parsed_dates.dt.year != data["original_air_year"]]
print(f"Year mismatches: {len(year_mismatch)}")

In [ ]:
# Check if id and number_in_series are identical (redundant column)
print(
    f"id == number_in_series for all rows: {(data['id'] == data['number_in_series']).all()}"
)

**Findings:**

- `original_air_date` is a string but parses cleanly to datetime.
  - can be converted during cleaning as it is easier to work with
- `original_air_year` is redundant with the year in `original_air_date` (0 mismatches)
  - we can drop this column and infer year from the date if needed
- `id` is identical to `number_in_series`
  - redundant column and can be dropped


## 6. Episode Coverage & Ordering


In [ ]:
# Episodes per season
season_counts = (
    data.groupby("season")
    .agg(
        episodes=("number_in_season", "count"),
        min_ep=("number_in_season", "min"),
        max_ep=("number_in_season", "max"),
    )
    .reset_index()
)
season_counts

In [ ]:
# Check for gaps in episode numbering within each season
has_gaps = False
for s, grp in data.groupby("season"):
    expected = set(range(1, grp["number_in_season"].max() + 1))
    actual = set(grp["number_in_season"])
    missing = expected - actual
    if missing:
        print(f"Season {s}: missing episode(s) {sorted(missing)}")
        has_gaps = True
if not has_gaps:
    print("No gaps found. All episode numbers are contiguous within each season.")

**Findings:**

- All 28 seasons are present with no gaps in episode numbering
- Season 28 only has 4 episodes (as noted, we will disregard this season in our analysis)
- Typical seasons have 20-25 episodes, however season 1 has only 13


## 7. Key Variable Distributions


### 7a. IMDb Ratings


In [ ]:
alt.Chart(data).mark_bar().encode(
    x=alt.X(
        "imdb_rating:Q", bin=alt.Bin(step=0.2, extent=[4, 10]), title="IMDb Rating"
    ),
    y=alt.Y("count()", title="Number of Episodes"),
    color=alt.value("#4c78a8"),
).properties(title="Distribution of IMDb Ratings", width=500, height=300)

In [ ]:
print(f"Mean: {data['imdb_rating'].mean():.2f}")
print(f"Median: {data['imdb_rating'].median():.2f}")
print(f"Std: {data['imdb_rating'].std():.2f}")
print(f"Range: {data['imdb_rating'].min():.1f} - {data['imdb_rating'].max():.1f}")

### 7b. US Viewers (millions)


In [ ]:
alt.Chart(data).mark_bar().encode(
    x=alt.X(
        "us_viewers_in_millions:Q",
        bin=alt.Bin(maxbins=30),
        title="US Viewers (millions)",
    ),
    y=alt.Y("count()", title="Number of Episodes"),
    color=alt.value("#f58518"),
).properties(title="Distribution of US Viewers", width=500, height=300)

In [ ]:
print(f"Mean: {data['us_viewers_in_millions'].mean():.2f}M")
print(f"Median: {data['us_viewers_in_millions'].median():.2f}M")
print(f"Std: {data['us_viewers_in_millions'].std():.2f}M")
print(
    f"Range: {data['us_viewers_in_millions'].min():.2f}M - {data['us_viewers_in_millions'].max():.2f}M"
)

### 7c. IMDb Votes


In [ ]:
alt.Chart(data).mark_bar().encode(
    x=alt.X("imdb_votes:Q", bin=alt.Bin(maxbins=30), title="IMDb Votes"),
    y=alt.Y("count()", title="Number of Episodes"),
    color=alt.value("#72b7b2"),
).properties(title="Distribution of IMDb Vote Counts", width=500, height=300)

In [ ]:
print(f"Mean: {data['imdb_votes'].mean():.0f}")
print(f"Median: {data['imdb_votes'].median():.0f}")
print(f"Range: {data['imdb_votes'].min():.0f} - {data['imdb_votes'].max():.0f}")

## 8. Temporal Patterns

A first look at how ratings and viewership have evolved. These patterns will be explored in depth during the visualization phase.


In [ ]:
# Average rating per season
alt.Chart(data).mark_line(point=True).encode(
    x=alt.X("season:O", title="Season", axis=alt.Axis(labelAngle=0)),
    y=alt.Y(
        "mean(imdb_rating):Q", title="Mean IMDb Rating", scale=alt.Scale(zero=False)
    ),
).properties(title="Average IMDb Rating per Season", width=600, height=300)

In [ ]:
# Average viewership per season
alt.Chart(data).mark_line(point=True, color="#f58518").encode(
    x=alt.X("season:O", title="Season", axis=alt.Axis(labelAngle=0)),
    y=alt.Y(
        "mean(us_viewers_in_millions):Q",
        title="Mean US Viewers (millions)",
        scale=alt.Scale(zero=False),
    ),
    color=alt.value("#f58518"),
).properties(title="Average US Viewership per Season", width=600, height=300)

In [ ]:
# Rating vs viewers, colored by season
alt.Chart(data).mark_circle(size=40, opacity=0.6).encode(
    x=alt.X("imdb_rating:Q", title="IMDb Rating", scale=alt.Scale(zero=False)),
    y=alt.Y("us_viewers_in_millions:Q", title="US Viewers (millions)"),
    color=alt.Color(
        "season:O",
        legend=alt.Legend(title="Season", columns=2),
        scale=alt.Scale(scheme="turbo"),
    ),
    tooltip=["title", "season", "imdb_rating", "us_viewers_in_millions"],
).properties(title="IMDb Rating vs US Viewers", width=500, height=400)

In [ ]:
# Day of the week episodes aired
data["air_date"] = pd.to_datetime(data["original_air_date"])
data["weekday"] = data["air_date"].dt.day_name()
data["weekday_abbr"] = data["air_date"].dt.strftime("%a")

weekday_order = ["Mon", "Tue", "Wed", "Thu", "Fri", "Sat", "Sun"]

alt.Chart(data).mark_bar().encode(
    x=alt.X(
        "weekday_abbr:N",
        sort=weekday_order,
        title="Day of the Week",
        axis=alt.Axis(labelAngle=0),
    ),
    y=alt.Y("count()", title="Number of Episodes"),
    color=alt.value("#54a24b"),
).properties(title="Episodes by Day of the Week", width=400, height=300)

## 9. Outliers & Notable Episodes


In [ ]:
cols = ["title", "season", "number_in_series", "imdb_rating", "us_viewers_in_millions"]

print("=== Top 5 by IMDb Rating ===")
display(data.nlargest(5, "imdb_rating")[cols])

print("\n=== Bottom 5 by IMDb Rating ===")
display(data.nsmallest(5, "imdb_rating")[cols])

In [ ]:
print("=== Top 5 by US Viewership ===")
display(data.nlargest(5, "us_viewers_in_millions")[cols])

print("\n=== Bottom 5 by US Viewership ===")
display(data.nsmallest(5, "us_viewers_in_millions")[cols])

## 10. Correlations


In [ ]:
# Correlation matrix for numeric columns relevant to our analysis
corr_cols = [
    "imdb_rating",
    "imdb_votes",
    "us_viewers_in_millions",
    "views",
    "season",
    "number_in_series",
]
corr = data[corr_cols].corr().round(2)
corr

In [ ]:
# Altair heatmap of correlations (lower triangle only)
import numpy as np

mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
corr_lower = corr.where(~mask)

corr_long = (
    corr_lower.reset_index()
    .melt(id_vars="index", var_name="variable", value_name="correlation")
    .dropna()
)
corr_long.columns = ["var1", "var2", "correlation"]

base = alt.Chart(corr_long).encode(
    x=alt.X("var1:N", title=None, sort=corr_cols),
    y=alt.Y("var2:N", title=None, sort=corr_cols),
)

heatmap = base.mark_rect().encode(
    color=alt.Color(
        "correlation:Q",
        scale=alt.Scale(scheme="blueorange", domain=[-1, 1]),
        title="Correlation",
    )
)

text = base.mark_text(fontSize=12).encode(
    text="correlation:Q",
    color=alt.condition(
        alt.datum.correlation > 0.6,
        alt.value("white"),
        alt.value("black"),
    ),
)

(heatmap + text).properties(title="Correlation Matrix", width=400, height=400)

## 11. Summary & Cleaning Roadmap

600 episodes across 28 seasons (1989 to 2016), no duplicates, no gaps in episode numbering. Both ratings and viewership decline over time. No suspicious outliers found.

### Cleaning Steps (for `02_cleaning.ipynb`)

1. **Drop season 28:** only 4 of 22 episodes present with most data missing. Keeping it would skew per-season aggregates and misrepresent the show's trajectory.
2. **Fill missing `us_viewers_in_millions` for season 8:** episodes 160, 161, 173 have all other data intact; viewer counts can be looked up manually.
3. **Parse `original_air_date`** to datetime; derive `weekday` column for the weekday-viewership analysis.
4. **Drop redundant/unused columns:**
   - `id` (identical to `number_in_series`)
   - `original_air_year` (derivable from `original_air_date`)
   - `image_url`, `video_url` (URLs, not needed for analysis)
   - `production_code` (internal Fox code, no analytical value)
   - `views` (likely website page views, not TV audience; scale doesn't match `us_viewers_in_millions`)
5. **Sort** by `number_in_series` (raw CSV is unordered).
6. **Save** cleaned dataset to `data/clean_data/simpsons_episodes_clean.csv`
